In [2]:
from typing import Literal
from pydantic import BaseModel
from langchain_openai import AzureChatOpenAI
import os
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
 
load_dotenv()

class CallSummary(BaseModel):
    employee_name: str
    employee_id: str
    agent_name: str
    summary: str
    sentiment: Literal["good", "bad", "normal"]

INSTRUCTIONS = """You are an information extraction assistant. Extract structured metadata from a service desk call transcript into the provided schema.

SCHEMA FIELDS (fill exactly as specified):
- employee_name: Full name of the caller/employee (e.g., "First Last"). If not present, return "".
- employee_id: The numeric employee identifier explicitly stated or confirmed during authentication. Digits only; strip spaces and punctuation. If not present, return "".
- agent_name: The primary service desk agent’s name (the person who greeted and handled the call). If missing or unclear, return "".
- summary: A concise 2–4 sentence, factual summary of the issue and outcome in neutral tone. Include the user’s goal, key systems/apps mentioned, steps guided by the agent, and the final result (resolved/unresolved/next steps).
- sentiment: One of {{"good","bad","normal"}} reflecting the caller’s overall sentiment at the end:
  - "good": resolved and/or caller expresses thanks/satisfaction.
  - "bad": unresolved issue, escalation/abandonment, or clear frustration.
  - "normal": neutral/mixed, outcome unclear, or insufficient evidence.

DECISION & NORMALIZATION RULES:
1) Names: Capitalize first and last names (e.g., "jose" → "Jose"). If multiple agents appear, select the one who greets and completes most of the interaction.
2) Employee ID: Prefer the numeric ID explicitly provided by the caller or verified by the agent. If an email-like string appears, do NOT use that—only the numeric ID. Keep digits only (e.g., "93 45-4472" → "93454472").
3) Evidence Priority: Prefer explicit statements from the transcript over assumptions. If a field is not explicitly present, set it to "" (except sentiment, which must be "good", "bad", or "normal").
4) Summary Content: Be specific about the system(s) (e.g., Workday, SSO portal) and the key action(s) (e.g., password reset, username format, URL used, MFA). Avoid small talk and filler. Mention resolution status.
5) Conflict Handling: If conflicting info exists (e.g., two names/IDs), prefer the most recently confirmed item by the agent and caller.
6) Safety: Do not introduce information not present in the transcript. Do not guess phone numbers, emails, or departments unless explicitly stated.
7) Noise & ASR Artifacts: Ignore stutters, repetitions, and filler; extract the clean underlying facts.

OUTPUT:
Return only the structured object matching the schema. The calling framework will handle JSON formatting.

TRANSCRIPT:
{transcript}
"""

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0,
    max_tokens=500,
    timeout=60,
)

prompt = ChatPromptTemplate.from_messages([("system", INSTRUCTIONS)])
structured_llm = llm.with_structured_output(CallSummary)
chain = prompt | structured_llm

In [12]:
result: CallSummary = chain.invoke({"transcript": transcript_text})

print(result)         # CallSummary(...)


employee_name='' employee_id='' agent_name='' summary='The caller interacted with the service desk to resolve an issue related to logging into a system. The agent guided the caller through entering their password and completing the login process. The issue was resolved successfully.' sentiment='good'


In [5]:
print(result.model_dump())  # JSON-like dict


{'employee_name': '', 'employee_id': '', 'agent_name': '', 'summary': 'The transcript provided does not contain any discernible content or details about the interaction, issue, or participants.', 'sentiment': 'normal'}


In [ ]:
print(result.summary) # Access fields directly

In [4]:

text = """Hi, thank you for calling Smithfield Service Desk. This is Jose. How can I help you? Yes Jose, I'm calling to try to reset my password. It's in my account may be locked. Is for workday or stamp or it's yes for work day. Work day. OK, sure. Let me assist with that. OK, let's begin. May I have please your full name? Jameel Robinson. OK, thank you so much. Could you please provide me your employee ID? 93454472. 93454472. Yes. OK. Thank you so much. And last may I have please a phone number? 252. 4323706. 3706 OK, thank you so much. OK uh, please allow me a couple of minutes in order to verify this information and help you with the password reset. OK, I'm gonna put you moment on hold and I'll be back. OK. Thank you. OK. Thank you so much for waiting in the line. OK, I find here your work day account. I see here that the account is active. So in this what what? What is the issue? You forget the password or? So I I'm trying to log in on I I could log in on my phone but I'm trying to log in on the computer and it's asking me to put in an e-mail address sowheniputinmyemailof93454472@sfi.cloud Yes, when I put in the password it's saying you may have entered the wrong e-mail address or the password is wrong, or your account might be locked. So understand OK. Go ahead, Sorry. Yeah. So that's what that's what I need help with. OK, when you let let me, could you please well try something when when you when you log in, you use you, you input the whole thing, the 93, the whole number at sfi.cloud or only the employee ID? Well see, on my phone I only use the employee ID, but on the computer it's asking me to put in the whole e-mail address. OK, Maybe we can try in the in the computer if you have the computer near to you, right or no, I got it. I got it right here with me. Yeah. Could you please try to access in the website myapps.smithfield.com? My apps. Dot smithfield.com OK, hold on. OK, in this moment you maybe were be able to see an login website. OK, so in the username and the username space I'll try to input only the employee ID, no da whole thing with the SFI, not only the deploy ID. OK. OK. And then put my password. Yeah, go Yeah. In this case, click on continue and input the other password. OK. It logs me in on here. OK. Thank you. No, you're welcome, Sir, by the moment. Is there anything else that I can do for you today or that will be all? That would be all OK. You're welcome. Thank you again so much for calling Smithfield Services. Have a great evening."""

result: CallSummary = chain.invoke({"transcript": text})

print(result)         # CallSummary(...)


employee_name='Jameel Robinson' employee_id='93454472' agent_name='Jose' summary="Jameel Robinson called the Smithfield Service Desk to reset his Workday password, reporting issues logging in on his computer. Agent Jose verified Jameel's identity and guided him to use only his employee ID (not the full email address) as the username at myapps.smithfield.com. After following these instructions, Jameel was able to log in successfully. The issue was resolved during the call." sentiment='good'


In [5]:
print(result.model_dump())  # JSON-like dict


{'employee_name': 'Jameel Robinson', 'employee_id': '93454472', 'agent_name': 'Jose', 'summary': "Jameel Robinson called the Smithfield Service Desk to reset his Workday password, reporting issues logging in on his computer. Agent Jose verified Jameel's identity and guided him to use only his employee ID (not the full email address) as the username at myapps.smithfield.com. After following these instructions, Jameel was able to log in successfully. The issue was resolved during the call.", 'sentiment': 'good'}


In [ ]:
print(result.summary) # Access fields directly

In [6]:
from dotenv import load_dotenv
import os

# Load .env from current directory
load_dotenv()

# Read the deployment/model name
model_name = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

print("Azure OpenAI model (deployment) from .env:", model_name)

Azure OpenAI model (deployment) from .env: gpt-4.1
